# 02 · Cleaning and features (Phase 2)

**Output:** `data/processed/coe_features.csv` — one row per car category and round, holding the target and every feature. All later notebooks read only this file.

**Target.** A dealer quotes at round *t*'s premium and secures the COE *h* rounds later:

$$R_{t,h} = P_{t+h} - P_t, \qquad r_{t,h} = \frac{P_{t+h}-P_t}{P_t}$$

We model the percentage rise *r* (premiums quadrupled since 2010) and convert back with $P_t$. Headline horizon *h* = 2 (about one month); *h* = 1, 3, 4 are sensitivity.

**Cleaning:** numbers from text; keep Cat A, B, E; sort by month then round; number rounds *t* = 0, 1, 2, … per category; flag the first round after the April–June 2020 suspension and **drop any target that spans it**; `post_2022` = 1 from May 2022.

In [1]:
import sys, json, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.config import *
pd.set_option("display.width", 180); pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

In [2]:
from src.data import load_raw, clean
from src.features import build_features, load_policy_events, FEATURES
panel = clean(load_raw())
events = load_policy_events()
feat = build_features(panel, events)
print(panel.shape, feat.shape)
print(panel.groupby("category").size().to_dict())
panel.loc[panel["after_gap"] == 1, ["category", "month", "bidding_no", "t"]]

(1194, 16) (1194, 57)
{'A': 398, 'B': 398, 'E': 398}


,category,month,bidding_no,t
246,A,2020-07,1,246
644,B,2020-07,1,246
1042,E,2020-07,1,246


## Features (only information available at round *t*)

| Feature | Definition |
|---|---|
| `bid_ratio`, `bid_ratio_lag1` | bids received ÷ quota at *t* and *t*−1 |
| `unsuccessful_share` | (bids received − successful) ÷ bids received |
| `quota_change_next_h` | % change from quota at *t* to *t+h*, used only while *t+h* is inside the quota quarter LTA has already announced (Feb–Apr, May–Jul, Aug–Oct, Nov–Jan); otherwise the last announced quota (end of the current quarter) |
| `quota_chg_6` | quota at *t* vs one quarter (6 rounds) earlier — the latest **announced** supply change (extra feature, see note) |
| `mom_1`, `mom_2`, `mom_4` | % change in premium over the last 1, 2, 4 rounds |
| `vol_12` | std. dev. of 1-round % changes over the last 12 rounds |
| `vol_ewma` | exponentially weighted volatility of 1-round % changes (λ = 0.94: the weight halves about every 11 rounds), used by the volatility-scaled model M6 (extra feature) |
| `e_minus_b` | (Cat E premium − Cat B premium) ÷ Cat B |
| `round_no`, `month_num` | round 1 or 2; calendar month |
| `event_next_h` | 1 if a policy event **already announced at t** takes effect in the next *h* rounds |
| `event_down_next_h`, `event_up_next_h` | the same, counting only events whose `direction` is down (or up) |
| `post_2022` | 1 from May 2022 |

*Note on `quota_change_next`:* LTA announces each quarter's quota about a week before the quarter starts, i.e. after the last round of the previous quarter. So the quota for *t+h* is known only when both rounds are in the same quarter, and the feature is often zero. We add `quota_chg_6`, the quarterly change already in force, as the supply signal a dealer does see.

In [3]:
cols = ["category", "month", "bidding_no", "premium", "quota", "bid_ratio", "mom_2", "vol_12",
        "quota_change_next_2", "quota_chg_6", "e_minus_b", "event_down_next_2", "event_up_next_2", "r_2", "R_2"]
feat[cols].dropna().tail(6)

,category,month,bidding_no,premium,quota,bid_ratio,mom_2,vol_12,quota_change_next_2,quota_chg_6,e_minus_b,event_down_next_2,event_up_next_2,r_2,R_2
1186,E,2026-07,1,129801,277,1.632,0.006,0.023,-0.051,0.131,-0.008,0,0,0.009,"1,199.000"
1187,E,2026-07,2,129971,263,1.711,0.008,0.022,0.000,0.091,0.001,0,0,0.039,"5,029.000"
1188,E,2026-08,1,131000,254,1.744,0.009,0.017,0.181,0.000,0.008,0,0,0.053,"6,890.000"
1189,E,2026-08,2,135000,254,1.839,0.039,0.013,0.059,-0.008,0.031,0,0,0.015,"2,000.000"
1190,E,2026-09,1,137890,300,1.710,0.053,0.013,-0.157,0.149,0.021,0,0,-0.006,-890.000
1191,E,2026-09,2,137000,269,1.602,0.015,0.014,0.004,0.027,0.030,0,0,-0.009,"-1,210.000"


## Check 1 — spot-check the target

In [4]:
row = feat[(feat.category == "B") & (feat.month == "2024-01") & (feat.bidding_no == 1)].iloc[0]
print(f"Cat B Jan 2024 round 1 premium S${row.premium:,.0f}; round 2 S${row.premium + row.R_1:,.0f}")
print(f"R_1 = S${row.R_1:,.0f}   (plan's check value S$26,990 -> this is the 1-round rise)")
print(f"R_2 = S${row.R_2:,.0f}   (2-round rise from the same quote)")
assert row.R_1 == 26990

Cat B Jan 2024 round 1 premium S$85,010; round 2 S$112,000
R_1 = S$26,990   (plan's check value S$26,990 -> this is the 1-round rise)
R_2 = S$17,328   (2-round rise from the same quote)


## Check 2 — recompute features by hand from rounds ≤ t only

In [5]:
rng = np.random.default_rng(5126)
sample = feat.dropna(subset=["vol_12"]).sample(4, random_state=1)
for _, r in sample.iterrows():
    past = panel[(panel.category == r.category) & (panel.t <= r.t)].sort_values("t")
    P = past["premium"].to_numpy()
    br = past["bids_received"].iloc[-1] / past["quota"].iloc[-1]
    mom2 = P[-1] / P[-3] - 1
    vol12 = pd.Series(P).pct_change().iloc[-12:].std()
    same = panel[panel.round_id == r.round_id].set_index("category")["premium"]
    emb = (same["E"] - same["B"]) / same["B"]
    ok = np.allclose([br, mom2, vol12, emb], [r.bid_ratio, r.mom_2, r.vol_12, r.e_minus_b])
    print(f"{r.category} {r.month} r{r.bidding_no}: bid_ratio {br:.3f}  mom_2 {mom2:+.3f}  "
          f"vol_12 {vol12:.3f}  e_minus_b {emb:+.3f}  -> match: {ok}")
    assert ok

E 2012-12 r1: bid_ratio 1.337  mom_2 +0.053  vol_12 0.038  e_minus_b +0.075  -> match: True
A 2018-12 r1: bid_ratio 1.159  mom_2 -0.164  vol_12 0.139  e_minus_b -0.005  -> match: True
E 2018-10 r2: bid_ratio 1.445  mom_2 +0.028  vol_12 0.043  e_minus_b +0.051  -> match: True
B 2026-06 r1: bid_ratio 1.462  mom_2 +0.006  vol_12 0.052  e_minus_b +0.016  -> match: True


## Check 3 — automated look-ahead test

Rebuild the features from a panel **truncated** at an earlier round. If any feature used a later round, its values before the truncation point would change.

In [6]:
cut = 250
feat_trunc = build_features(panel[panel.round_id <= cut], events)
check_cols = [c for c in FEATURES]
a = feat.set_index(["category", "t"]).loc[feat_trunc.set_index(["category", "t"]).index, check_cols]
b = feat_trunc.set_index(["category", "t"])[check_cols]
max_diff = (a - b).abs().max()
print(max_diff)
assert (max_diff.fillna(0) < 1e-12).all(), "a feature uses later rounds"
print("No look-ahead in", len(check_cols), "features")

bid_ratio            0.000
bid_ratio_lag1       0.000
unsuccessful_share   0.000
mom_1                0.000
mom_2                0.000
mom_4                0.000
vol_12               0.000
vol_ewma             0.000
quota_chg_6          0.000
e_minus_b            0.000
round_no             0.000
month_num            0.000
post_2022            0.000
dtype: float64
No look-ahead in 13 features


In [7]:
out = feat.drop(columns=["date"]).copy()
out.to_csv(PROC / "coe_features.csv", index=False)
print("saved", PROC / "coe_features.csv", out.shape)
feat[FEATURES + ["r_1", "r_2", "r_3", "r_4"]].describe().T

saved /Users/khushali/Documents/IS5126_Handson/Project/coe-guarantee/data/processed/coe_features.csv (1194, 56)


,count,mean,std,min,25%,50%,75%,max
bid_ratio,"1,194.000",1.565,0.293,0.806,1.358,1.504,1.687,4.063
bid_ratio_lag1,"1,191.000",1.565,0.293,0.806,1.358,1.505,1.687,4.063
unsuccessful_share,"1,194.000",0.353,0.107,0.077,0.272,0.347,0.422,0.761
mom_1,"1,191.000",0.007,0.070,-0.373,-0.028,0.007,0.037,0.522
mom_2,"1,188.000",0.015,0.103,-0.375,-0.038,0.011,0.063,0.776
mom_4,"1,182.000",0.028,0.144,-0.396,-0.055,0.020,0.102,1.187
vol_12,"1,158.000",0.062,0.030,0.013,0.042,0.054,0.072,0.170
vol_ewma,"1,158.000",0.067,0.023,0.029,0.051,0.061,0.076,0.165
quota_chg_6,"1,176.000",0.010,0.161,-0.523,-0.072,0.000,0.086,0.682
e_minus_b,"1,194.000",0.018,0.035,-0.064,-0.002,0.010,0.031,0.251
